In [ ]:
# ======================================================================================
# 🏥 의료 AI Agent 데이터셋 생성기 [FINAL VERSION!!]
# - PII 마스킹 (Hugging FaceD에서 PII 모델 로드함)
# - 5가지 핵심 시나리오 & 5개 도구 (1개는 예비로 남겨둠)
# - 오류 복구 시나리오 및 로직 포함
# - 깔끔한 로그 출력
# ======================================================================================
import os
import sys
import json
import requests
import warnings
# 불필요한 경고 메시지 제거
warnings.filterwarnings("ignore")
from datetime import datetime
from pathlib import Path
from openai import OpenAI

# --------------------------------------------------------------------------------------
# 0. 초기 설정 및 .env 로드
# --------------------------------------------------------------------------------------
try:
    from dotenv import load_dotenv
    load_dotenv() 
except ImportError:
    pass

OUTPUT_DIR = Path("output")
OUTPUT_DIR.mkdir(exist_ok=True)

SOLAR_API_KEY = os.getenv("SOLAR_API_KEY", "")
SERVICE_KEY = os.getenv("SERVICE_KEY", "")

if SOLAR_API_KEY:
    client = OpenAI(api_key=SOLAR_API_KEY, base_url="https://api.upstage.ai/v1")
else:
    client = None
    print("🚨 SOLAR_API_KEY가 없습니다. .env 파일을 확인하세요.")
    sys.exit(1)

# evaluate.py 로드
if str(Path.cwd()) not in sys.path:
    sys.path.insert(0, str(Path.cwd()))
try:
    from evaluate import evaluate_file
except ImportError:
    def evaluate_file(input_path, output_path): pass

# --------------------------------------------------------------------------------------
# 1. PII 마스킹 (Hugging Face)
# --------------------------------------------------------------------------------------
pii_classifier = None
USE_PII_MASKING = False

def load_pii_model():
    """Hugging Face 모델 로드"""
    global pii_classifier, USE_PII_MASKING
    try:
        import torch
        from transformers import pipeline
        
        print("🔄 PII 모델 준비 중...", end=" ")
        device = 0 if torch.cuda.is_available() else -1
        
        pii_classifier = pipeline(
            "token-classification",
            model="seungkukim/korean-pii-masking",
            aggregation_strategy="simple",
            device=device
        )
        USE_PII_MASKING = True
        print("✅ 완료")
        
    except Exception as e:
        print(f"\n⚠️ PII 로드 실패 (기능 OFF): {e}")
        USE_PII_MASKING = False

def mask_pii(text: str) -> str:
    """개인정보 마스킹"""
    if not text or not USE_PII_MASKING: return text
    
    try:
        results = pii_classifier(text)
        masked_text = text
        # 인덱스 밀림 방지 (Reverse)
        for entity in sorted(results, key=lambda x: x['start'], reverse=True):
            start, end = entity['start'], entity['end']
            entity_type = entity.get('entity_group', 'UNKNOWN')
            original = text[start:end]
            
            if entity_type == "PS_NAME": mask = original[0] + "*" * (len(original) - 1)
            elif "PHONE" in entity_type or "MOBILE" in entity_type: mask = "[전화번호]"
            elif "CARD" in entity_type: mask = "[카드번호]"
            elif "RESIDENT" in entity_type: mask = "[주민번호]"
            else: mask = "*" * len(original)
            
            masked_text = masked_text[:start] + mask + masked_text[end:]
        return masked_text
    except:
        return text

# --------------------------------------------------------------------------------------
# 2. 도구(Tools) 정의 (5개)
# --------------------------------------------------------------------------------------
TOOLS = [
    {"type": "function", "function": {"name": "get_emergency_room_list", "description": "지역(시도, 시군구)을 입력받아 실시간 응급실 목록 조회. 결과가 없으면 빈 리스트 반환.", "parameters": {"type": "object", "properties": {"STAGE1": {"type": "string"}, "STAGE2": {"type": "string"}}, "required": ["STAGE1", "STAGE2"]}}},
    {"type": "function", "function": {"name": "get_hospital_basic_info_by_hpid", "description": "병원 ID(HPID)로 뇌출혈/뇌경색 등 중증질환 수용 가능 여부 조회.", "parameters": {"type": "object", "properties": {"HPID": {"type": "string"}}, "required": ["HPID"]}}},
    {"type": "function", "function": {"name": "get_trauma_center_list", "description": "권역외상센터 목록 조회.", "parameters": {"type": "object", "properties": {"Q0": {"type": "string"}, "Q1": {"type": "string"}}, "required": ["Q0", "Q1"]}}},
    {"type": "function", "function": {"name": "get_er_message_by_hpid", "description": "병원 ID로 응급실 실시간 공지사항(수용 불가 메시지 등) 조회.", "parameters": {"type": "object", "properties": {"HPID": {"type": "string"}}, "required": ["HPID"]}}},
    {"type": "function", "function": {"name": "get_travel_time", "description": "출발지에서 병원까지의 예상 이동 시간 계산.", "parameters": {"type": "object", "properties": {"origin": {"type": "string"}, "destination": {"type": "string"}}, "required": ["origin", "destination"]}}}
]

# --------------------------------------------------------------------------------------
# 3. 도구 실행기 (Hybrid Mock)
# --------------------------------------------------------------------------------------
def execute_tool_call(name, args):
    # 1. 응급실 목록 (오류 복구 로직)
    if name == "get_emergency_room_list":
        stage2 = args.get("STAGE2", "")
        # [Fail] 계양구 -> 빈 리스트
        if "계양구" in stage2:
            return json.dumps({"status": "OK", "items": [], "message": "해당 지역 결과 없음"}, ensure_ascii=False)
        # [Success] 부평구 -> 데이터 있음
        if "부평구" in stage2:
            return json.dumps({
                "status": "OK", 
                "items": [
                    {"dutyName": "부평세림병원", "hpid": "A1200005", "hvec": "5"},
                    {"dutyName": "인천성모병원", "hpid": "A1200001", "hvec": "2"}
                ]
            }, ensure_ascii=False)
        # 기본 Mock
        return json.dumps({"status": "OK", "items": [{"dutyName": f"{stage2} 한국병원", "hpid": "A99", "hvec": "3"}]}, ensure_ascii=False)

    # 2. 병원 기본 정보
    if name == "get_hospital_basic_info_by_hpid":
        return json.dumps({"status": "OK", "items": [{"dutyName": "삼성서울병원", "MKioskTy1": "Y"}]}, ensure_ascii=False)

    # 3. 외상 센터
    if name == "get_trauma_center_list":
        return json.dumps({"status": "OK", "items": [{"dutyName": "아주대병원 권역외상센터", "hpid": "A123"}]}, ensure_ascii=False)

    # 4. 응급실 메시지
    if name == "get_er_message_by_hpid":
        hpid = args.get("HPID", "")
        msg = "현재 심근경색 시술 불가 / 소아 진료 불가" if hpid == "A1100008" else "정상 운영"
        return json.dumps({"status": "OK", "items": [{"symBlkMsg": msg}]}, ensure_ascii=False)

    # 5. 이동 시간
    if name == "get_travel_time":
        dest = args.get("destination", "")
        return json.dumps({"status": "OK", "time_minutes": 15 + (len(dest)%10), "distance_km": 3.5}, ensure_ascii=False)
        
    return json.dumps({"status": "ERROR"}, ensure_ascii=False)

# --------------------------------------------------------------------------------------
# 4. 시나리오 생성기
# --------------------------------------------------------------------------------------
def generate_dynamic_scenario(s_id, turns, meta):
    print(f"🔄 처리 중: {s_id} ({meta['task']})")
    
    messages = [{"role": "system", "content": "You are a helpful medical AI agent."}]
    conversation_log = []

    for user_text in turns:
        masked_input = mask_pii(user_text)
        messages.append({"role": "user", "content": masked_input})
        conversation_log.append({"role": "user", "content": masked_input})

        # Loop (Max 3)
        for i in range(3):
            try:
                resp = client.chat.completions.create(
                    model="solar-pro",
                    messages=messages,
                    tools=TOOLS,
                    tool_choice="auto"
                )
                ai_msg = resp.choices[0].message
            except Exception as e:
                print(f"  ❌ API Error: {e}")
                return None

            if ai_msg.tool_calls:
                tc_data = [{"id": tc.id, "type": "function", "function": {"name": tc.function.name, "arguments": tc.function.arguments}} for tc in ai_msg.tool_calls]
                asst_msg = {"role": "assistant", "content": ai_msg.content, "tool_calls": tc_data}
                messages.append(asst_msg)
                conversation_log.append(asst_msg)

                for tc in ai_msg.tool_calls:
                    args = json.loads(tc.function.arguments)
                    result = execute_tool_call(tc.function.name, args)
                    
                    tool_msg = {"role": "tool", "tool_call_id": tc.id, "content": result}
                    messages.append(tool_msg)
                    conversation_log.append(tool_msg)
            else:
                raw_content = ai_msg.content or ""
                masked_output = mask_pii(raw_content)
                
                asst_msg = {"role": "assistant", "content": masked_output}
                messages.append(asst_msg)
                conversation_log.append(asst_msg)
                break 

    return {
        "id": s_id,
        "row": {
            "messages": conversation_log,
            "metadata": json.dumps({"tools": TOOLS}, ensure_ascii=False)
        },
        "meta_info": meta
    }

# --------------------------------------------------------------------------------------
# 5. 메인 실행
# --------------------------------------------------------------------------------------
if __name__ == "__main__":
    load_pii_model()

    # ✅ 사용자 정의 5대 시나리오 (최종 수정본)
    TARGET_SCENARIOS = [
        # 1. 추천 (김철수 예시)
        {
            "id": "S1_Rec", 
            "turns": ["내 이름은 김철수고 내 전화번호는 010-1234-5678이야. 배가 너무 아픈데 미추홀구 쪽 응급실 추천해주고, 가능한 응급실에서 내 번호로 전화해주면 좋겠어."], 
            "meta": {"type": "Single", "task": "Recommendation"}
        },
        # 2. 복합 필터링 (폭설/도로 상황)
        {
            "id": "S2_Filter", 
            "turns": ["폭설 떄문에 도로가 많이 막힌 상황인데, 서울 성북구/도봉구/강북구 쪽에서 지금 정상 운영 중인 응급실만 알려줘. 임시 폐쇄되었거나 수용 불가능한 곳은 뺴줘."], 
            "meta": {"type": "Single", "task": "Complex Filtering"}
        },
        # 3. 거리 비교
        {
            "id": "S3_Compare", 
            "turns": ["미추홀구에 있는 응급실 목록 알려줘.", "주안역에서 제일 가까운 곳은 어디야?"], 
            "meta": {"type": "Multi", "task": "Tool Composition"}
        },
        # 4. 오류 복구 (계양구 -> 부평구)
        {
            "id": "S4_Recovery", 
            "turns": ["인천 계양구에 있는 응급실 찾아줘.", "거기 없으면 바로 옆인 부평구로 다시 찾아봐줘."], 
            "meta": {"type": "Multi", "task": "Error Recovery"}
        },
        # 5. 중증 확인 (삼성서울병원 메시지)
        {
            "id": "S5_Critical", 
            "turns": ["강남구 심근경색 응급실 급해.", "삼성서울병원은 지금 환자 받는지 확인해주고 없으면 강남구 말고 가장 가까운 곳에서 심근경색 환자 받는 응급실 확인해줘."], 
            "meta": {"type": "Multi", "task": "Critical Condition Check"}
        }
    ]

    generated_data = []
    print(f"\n🚀 시나리오 생성 시작 ({len(TARGET_SCENARIOS)}개)...")

    for target in TARGET_SCENARIOS:
        res = generate_dynamic_scenario(target["id"], target["turns"], target["meta"])
        if res: generated_data.append(res)

    # 저장
    dataset_path = OUTPUT_DIR / "final_dataset.jsonl"
    with open(dataset_path, "w", encoding="utf-8") as f:
        for entry in generated_data:
            f.write(json.dumps(entry, ensure_ascii=False) + "\n")

    print(f"✅ 저장 완료: {dataset_path}")

    # 평가 실행
    print("\n📊 BFCL 평가 실행 중...")
    eval_result_path = OUTPUT_DIR / "final_evaluation.json"
    evaluate_file(str(dataset_path), str(eval_result_path))
    
    print("\n🎉 완료되었습니다!")

🔄 PII 모델 준비 중... 

Device set to use cpu


✅ 완료

🚀 시나리오 생성 시작 (5개)...
🔄 처리 중: S1_Rec (Recommendation)
🔄 처리 중: S2_Filter (Complex Filtering)
🔄 처리 중: S3_Compare (Tool Composition)
🔄 처리 중: S4_Recovery (Error Recovery)
🔄 처리 중: S5_Critical (Critical Condition Check)
✅ 저장 완료: output\final_dataset.jsonl

📊 BFCL 평가 실행 중...

📊 평가 결과 요약
총 평가 항목: 5
통과: 5 (100.0%)
실패: 0 (0.0%)

📋 평가 항목별 통과율:
  - Correct Function Name: 5/5
  - Valid Arguments: 5/5
  - No Hallucinated Calls: 5/5

✅ 상세 결과 저장: output\final_evaluation.json

🎉 완료되었습니다!
